# 03 · The upset model and the critical gap
### Chess upsets over the board (DA2402 course project)

This notebook answers the project question: how large must the rating gap be before an underdog's win (or draw) is an upset, and does that gap depend on the time control? It starts from the clean table written by 02_cleaning and fits the model.

| Section | Content |
|---|---|
| 1 | the games used |
| 2 | why the textbook Elo formula cannot define an upset (observed score vs Elo expectation, draw rates, White's advantage) |
| 3 | the definition of an upset and the model, with one worked case |
| 4 | fitting the model: 3 time controls × 2 underdog colours |
| 5 | the critical gaps at three strength levels and four thresholds, with 95 % intervals |
| 6 | do the critical gaps differ between time controls? (differences with intervals) |
| 7 | does the model predict tournaments it has never seen? (held-out calibration) |
| 8 | robustness: verified ratings, sensitivity run, a model-free (isotonic) curve, and a model with a rating × gap interaction |
| 9 | expected vs observed upsets by strength level, for the main and the interaction model |
| 10 | the ordering check classical < rapid < blitz |
| 11 | the upsets under the final definition: counts, audit, repeat giant-killers |
| 12 | output files |

**Run:** *Kernel > Restart & Run All* after notebook 02. On the full data (864 thousand games) the notebook needs about 1-2 minutes and at most about 2.5 GB of memory. Every section after the configuration cells reads its inputs from disk, so a section can be re-run on its own.

**Outputs** (in `clean/`, the same file names and columns as the project's earlier scripts so that later code keeps working): `upsets_empirical.parquet` (per-game probabilities and upset flags; `_verified`, `_sens`, `_interaction` for the other runs), `critical_gap_empirical.csv`, `critical_gap_diff_empirical.csv`, `critical_gap_decisive.csv`, `critical_gap_isotonic.csv`, `fitted_curves.csv`, `model_coefficients_empirical.csv`, `calibration_empirical.csv` (each also `_verified`, `_sens`; the interaction run writes `_interaction`), `kappa_table.csv`, `curve_bins.csv`, `expected_upsets.csv`, `model_fit_by_band.csv`, `ordering_check.csv`, `giant_killers.csv`, `upset_audit_top.csv`. The small CSV files and the text reports are copied to `reports/model_outputs/`.

## 0. Configuration

In [1]:
import os, time, shutil, platform, datetime, warnings
import numpy as np, pandas as pd, pyarrow.parquet as pq, sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 120)

DATA_ROOT = os.environ.get("CHESS_DATA_ROOT", "../data")          # folders raw/, interim/, clean/ below it
REPORTS   = os.environ.get("CHESS_REPORTS", "../reports")
TABLE     = os.environ.get("CHESS_MODEL_TABLE", os.path.join(DATA_ROOT, "clean", "analysis_games.parquet"))   # input of this notebook
OUT       = os.environ.get("CHESS_MODEL_OUT", os.path.join(DATA_ROOT, "clean"))                              # results are written here
SMALL_OUT = os.path.join(REPORTS, "model_outputs")                 # copies of the small CSV files and the text reports

ALPHAS   = (0.20, 0.10, 0.05, 0.01)                                # upset thresholds; 0.10 is the headline value
REF_ELO  = {"<2200": 2100, "2200-2499": 2350, "2500+": 2600}       # reference average rating of each strength band
GRID     = np.arange(0, 1505, 5)                                   # gaps at which the curves are evaluated (5-point grid)
CAP      = 900                                                     # gaps above 900 are treated as 900 in the model
B_DRAWS  = 1000                                                    # parameter draws for the 95 % intervals
SEED_CI, SEED_CURVES = 7, 11                                       # numpy default_rng seeds: intervals / plotted bands (as in the original script)
NEAR     = 25                                                      # "near-equal" games for draw rates: gap <= 25
CLASSES, COLOURS = ("classical", "rapid", "blitz"), ("white", "black")
os.makedirs(OUT, exist_ok=True); os.makedirs(SMALL_OUT, exist_ok=True)
def say(s): display(Markdown(s))
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | scikit-learn {sklearn.__version__} | run on {datetime.date.today()}")
print("input table =", os.path.abspath(TABLE)); print("outputs     =", os.path.abspath(OUT))

python 3.13.5 | pandas 3.1.0 | numpy 2.5.4 | scikit-learn 1.9.1 | run on 2026-11-05
input table = C:\dev\chess-upsets\submission\data\clean\analysis_games.parquet
outputs     = C:\dev\chess-upsets\submission\data\clean


## 1. The games used

Only the columns the model needs are read. `verified` (both PGN ratings confirmed by the FIDE list within 100 points) and `short_game` (fewer than 20 half-moves) were stored by notebook 02; they are recomputed here from their definitions and checked against the stored columns, so that the notebook also runs on an older table that does not have them.

In [2]:
COLS = ["game_url", "broadcast_name", "round", "white", "black", "white_fide_id", "black_fide_id", "period", "year", "tc_class", "tc_borderline",
        "n_plies", "termination", "score_white", "white_elo_final", "black_elo_final", "white_elo_fide", "black_elo_fide", "elo_mismatch_any",
        "gap", "avg_elo", "has_underdog", "underdog_color", "underdog_score", "band", "gap_bin", "verified", "short_game"]
_have = set(pq.ParquetFile(TABLE).schema_arrow.names)
games = pd.read_parquet(TABLE, columns=[c for c in COLS if c in _have])
_ok = lambda s: games[f"{s}_elo_fide"].notna() & ((games[f"{s}_elo_final"] - games[f"{s}_elo_fide"]).abs() <= 100)
_ver, _short = (_ok("white") & _ok("black")).to_numpy(), (games["n_plies"] < 20).to_numpy()
if "verified" in games: assert (games["verified"].to_numpy() == _ver).all(), "stored verified flag differs from its definition"
if "short_game" in games: assert (games["short_game"].to_numpy() == _short).all(), "stored short_game flag differs from its definition"
games["verified"], games["short_game"] = _ver, _short
assert games["game_url"].is_unique

In [3]:
u_all = games[games["has_underdog"]]
tab = u_all.groupby(["tc_class", "underdog_color"]).size().unstack().reindex(list(CLASSES))
tab["total"] = tab.sum(axis=1)
display(tab)
say(f"**{len(games):,}** analysis games, of which **{len(u_all):,}** have an underdog (rating gap > 0; {int((~games.has_underdog).sum()):,} exact ties are not used by the model). "
    f"Both ratings are confirmed by the FIDE list in **{u_all.verified.mean():.1%}** of them. "
    f"Tournaments (broadcast name × year): **{(u_all.broadcast_name.astype(str) + '|' + u_all.period.astype(str).str[:4]).nunique():,}**.")

underdog_color,black,white,total
tc_class,,,
classical,309205,303554,612759
rapid,71894,70635,142529
blitz,53940,53371,107311


**863,971** analysis games, of which **862,599** have an underdog (rating gap > 0; 1,372 exact ties are not used by the model). Both ratings are confirmed by the FIDE list in **76.1%** of them. Tournaments (broadcast name × year): **10,106**.

## 2. Why not simply use the Elo formula?

Elo's formula gives the expected score of the lower-rated player (the *underdog*) from the rating gap alone: E = 1 / (1 + 10^(gap/400)), where a win counts 1, a draw ½ and a loss 0. It would be the obvious yardstick for "surprising" results. Three facts from the data rule it out as the definition of an upset.

### 2.1 Observed score against the Elo expectation

In [4]:
u = games[games["has_underdog"]].copy()
u["elo_expected"] = 1 / (1 + 10 ** (u["gap"] / 400))
ov = u.groupby("tc_class").agg(games=("gap", "size"), observed_score=("underdog_score", "mean"), elo_expected=("elo_expected", "mean")).reindex(list(CLASSES))
ov["observed_minus_elo"] = ov.observed_score - ov.elo_expected
display(ov.round(4))
u["gap100"] = (np.minimum(u["gap"], 600) // 100 * 100).astype(int)
bg = u.groupby(["tc_class", "gap100"]).agg(games=("gap", "size"), observed=("underdog_score", "mean"), elo=("elo_expected", "mean"))
bg["diff_pts"] = 100 * (bg.observed - bg.elo)
bgw = bg.unstack("tc_class").reindex(columns=list(CLASSES), level=1)
bgw.index = [f"{g}-{g + 99}" if g < 600 else "600+" for g in bgw.index]; bgw.index.name = "gap"
display(bgw[["games", "diff_pts"]].round(1))
say(f"In every time control the underdog scores **more** than Elo predicts (classical {100 * ov.loc['classical', 'observed_minus_elo']:+.1f}, "
    f"rapid {100 * ov.loc['rapid', 'observed_minus_elo']:+.1f}, blitz {100 * ov.loc['blitz', 'observed_minus_elo']:+.1f} percentage points on average), "
    "and the excess grows with the gap (second table: observed minus Elo score in points per 100 games). The formula is not calibrated for these games.")

,games,observed_score,elo_expected,observed_minus_elo
tc_class,,,,
classical,612759,0.3222,0.2869,0.0353
rapid,142529,0.3031,0.2550,0.0481
blitz,107311,0.3206,0.2662,0.0544


games                diff_pts            
tc_class classical  rapid  blitz classical rapid blitz
gap                                                   
0-99        189909  34995  29389       1.7   2.2   2.1
100-199     206072  40969  32194       4.0   4.8   5.8
200-299     119421  30928  21814       4.8   6.4   7.5
300-399      56074  17192  11709       5.0   6.3   8.3
400-499      24089   9204   6161       4.3   5.9   7.1
500-599      10260   4581   3090       4.0   5.7   5.9
600+          6934   4660   2954       3.2   5.9   4.6

In every time control the underdog scores **more** than Elo predicts (classical +3.5, rapid +4.8, blitz +5.4 percentage points on average), and the excess grows with the gap (second table: observed minus Elo score in points per 100 games). The formula is not calibrated for these games.

### 2.2 Draws

Elo's expected score mixes wins and draws into one number, but a draw by a weak player against a much stronger one is a very different event from a win, and the share of draws depends strongly on the time control and on the players' strength. The table shows the draw rate among near-equal games (gap ≤ 25 points) by time control and strength band (`kappa` is the draw parameter of the Elo-Davidson model, 2p/(1−p); it is stored for the dashboard), and the draw rate over all games.

In [5]:
near = games[games["gap"] <= NEAR]
def kappa_from(df):
    p = (df["score_white"] == 0.5).mean()
    return 2 * p / (1 - p), p, len(df)
k_class = {c: kappa_from(d) for c, d in near.groupby("tc_class")}
rows = []
for (c, b), d in near.groupby(["tc_class", "band"]):
    k = kappa_from(d) if len(d) >= 500 else k_class[c]                  # a cell needs 500 near-equal games, otherwise the class value is used
    rows.append(dict(tc_class=c, band=b, n_near_equal=len(d), draw_rate_equal=k[1], kappa=k[0], fallback=len(d) < 500))
ktab = pd.DataFrame(rows)
ktab.to_csv(os.path.join(OUT, "kappa_table.csv"), index=False)
dr = games.assign(draw=games.score_white == 0.5).pivot_table(index="band", columns="tc_class", values="draw", aggfunc="mean")[list(CLASSES)]
show = ktab.pivot(index="band", columns="tc_class", values="draw_rate_equal")[list(CLASSES)].add_prefix("near-equal ").join(dr.add_prefix("all games "))
display((100 * show).round(1).loc[["<2200", "2200-2499", "2500+"]])
say(f"Near-equal classical games between players rated 2500+ end in a draw **{100 * ktab.set_index(['tc_class', 'band']).loc[('classical', '2500+'), 'draw_rate_equal']:.0f} %** of the time, "
    f"near-equal blitz games below 2200 only **{100 * ktab.set_index(['tc_class', 'band']).loc[('blitz', '<2200'), 'draw_rate_equal']:.0f} %**. "
    "A single expected score cannot say whether a given draw or win was surprising; the model below therefore treats wins and draws separately.")

tc_class,near-equal classical,near-equal rapid,near-equal blitz,all games classical,all games rapid,all games blitz
band,,,,,,
<2200,32.7,20.7,13.9,25.2,15.0,10.6
2200-2499,49.1,35.5,23.2,36.7,24.2,16.1
2500+,62.9,43.9,29.3,53.6,38.9,25.4


Near-equal classical games between players rated 2500+ end in a draw **63 %** of the time, near-equal blitz games below 2200 only **14 %**. A single expected score cannot say whether a given draw or win was surprising; the model below therefore treats wins and draws separately.

### 2.3 Who has White

In [6]:
m = u[(u.gap >= 50) & (u.gap < 200)]
wa = m.groupby(["tc_class", "underdog_color"])["underdog_score"].agg(games="size", mean_score="mean").unstack("underdog_color").reindex(list(CLASSES))
display(wa.round(4))
d_ = (wa[("mean_score", "white")] - wa[("mean_score", "black")]) * 100
say(f"At gaps of 50–199 points an underdog with **White** scores {d_.min():.1f}–{d_.max():.1f} points per 100 games more than with Black. "
    "Elo's formula ignores colour; the model is therefore fitted **separately for White and Black underdogs**.")
del u, m

games         mean_score        
underdog_color   black   white      black   white
tc_class                                         
classical       155352  154316     0.3277  0.4070
rapid            29841   29516     0.3374  0.4047
blitz            23928   23728     0.3547  0.4036

At gaps of 50–199 points an underdog with **White** scores 4.9–7.9 points per 100 games more than with Black. Elo's formula ignores colour; the model is therefore fitted **separately for White and Black underdogs**.

## 3. What counts as an upset: definition and model

### 3.1 The idea

A result is an upset when it was *unlikely* for a game like this one. "Like this one" means: the same rating gap, the same time control, the same colour of the underdog and players of a similar strength. "Unlikely" means: a model fitted to the games themselves gives the result a probability below a threshold α. We use α = 0.10 (a one-in-ten result) as the headline value and report 0.20, 0.05 and 0.01 too.

### 3.2 The model

For each of the 6 slices (3 time controls × 2 underdog colours) two separate logistic regressions are fitted on the games of that slice:

* P(win): the probability that the underdog wins, and
* P(not lose): the probability that the underdog wins *or* draws.

A logistic regression models a probability *p* through its *log-odds* (logit), log(p / (1 − p)), as a linear function of the inputs; turning the linear value *z* back into a probability uses p = 1 / (1 + e^(−z)), which always lies between 0 and 1. Our linear function is

  logit P = b0 + b1·x + b2·x² + b3·x³ + b4·e,  with x = min(gap, 900) / 300 and e = (average rating − 2300) / 100.

1. x, x², x³ (a cubic in the gap) let the curve bend: the probability falls quickly at first and then flattens out, a shape a straight line in x cannot follow.
2. Capping the gap at 900 stops the few games with extreme gaps (there is too little data beyond 900) from steering the curve; beyond 900 the curve stays flat. Dividing by 300 and by 100 only keeps the numbers near 1, which makes the fit numerically stable; it does not change the fitted curve.
3. e, the average rating of the two players, lets the same gap mean different things for weaker and stronger players (strong players convert advantages more reliably). It enters as one straight-line term: this is the main simplification of the model, examined in section 8.4.
4. The coefficients b0 to b4 are found by maximum likelihood (the values that make the observed wins and losses most probable) with scikit-learn's `LogisticRegression(C=inf)`, i.e. *without* the penalty scikit-learn adds by default, because we want the plain estimate, not a shrunken one.

Why not a draw model (Elo-Davidson)? An earlier version of the project fitted one (`fit_model.py`): it got the underdog's *score* right but gave too few wins and too many draws at large gaps in classical chess. Fitting P(win) and P(not lose) directly needs no assumption about how draws arise.

### 3.3 From probabilities to an upset flag

For each game the fitted model gives P(win) and P(not lose) at that game's own gap and average rating (P(not lose) is never allowed below P(win)). The probability of a result at least as good as the one that happened, for the underdog, is

* q = P(win) if the underdog won,
* q = P(not lose) if the underdog drew (a draw *or better*),
* q = 1 if the underdog lost (a loss is never a surprise).

The game is an upset at level α if the underdog did not lose and q < α.

### 3.4 The critical gap

The critical gap is the smallest gap at which the underdog's chance of winning falls below α: the first point of a 5-point grid (0, 5, 10, and so on) where the fitted P(win) < α. Because the curve depends on the players' strength, it is evaluated at three reference ratings, one per strength band: 2100 (band < 2200), 2350 (2200-2499) and 2600 (2500+). "Not reached" means that the curve stays above α at every gap (for α = 0.01 in rapid and blitz the underdog still wins more than 1 % of the time even at the largest gaps in the data).

### 3.5 How uncertain is a critical gap? Clustered intervals

Games are not independent: the games of one tournament share the venue, the conditions, the field and often the same rating errors. Treating 600,000 games as independent would make the intervals much too narrow. The tournament (broadcast name and year; about 10,000 of them) is therefore the unit of independence:

* the covariance of the coefficients is the cluster-robust "sandwich" V = A⁻¹ (Σ_t s_t s_tᵀ) A⁻¹ · T/(T−1), where A = Σ_i p_i(1−p_i) x_i x_iᵀ is the usual information matrix (the "bread"), s_t = Σ_{i in tournament t} (y_i − p_i) x_i is the score of tournament t (the "meat"), and T is the number of tournaments;
* 1,000 coefficient vectors are drawn from the normal distribution N(b̂, V); each draw gives its own curve and its own critical gap; the 2.5 % and 97.5 % points of these 1,000 critical gaps are the 95 % interval;
* the random draws use numpy's `default_rng(7)` (seed fixed so that every run gives the same intervals; the shaded bands of the stored curves use `default_rng(11)` and 500 draws). A *difference* between two time controls is computed draw by draw, so the uncertainty of both fits is carried through.

### 3.6 Is the model right? Held-out tournaments

The tournaments are split into two halves by a hash of the tournament name and year (so all games of one event fall on the same side, and the split is the same on every computer). The model is fitted on one half and its predictions are compared with the other half (section 7). The final model is then refitted on all games.

The functions below implement sections 3.2-3.6. `run_model` performs one complete run (fits, calibration, critical gaps with intervals, differences, curves, decisive-game and isotonic cross-checks, per-game flags) and writes its files; the robustness runs of section 8 call the same function.

In [7]:
def design(gap, avg, interaction=False):
    """Model inputs: x, x^2, x^3, e  (+ x*e, x^2*e for the interaction model);  x = min(gap, 900)/300, e = (avg - 2300)/100."""
    x = np.minimum(np.asarray(gap, float), CAP) / 300
    e = (np.asarray(avg, float) - 2300) / 100
    cols = [x, x ** 2, x ** 3, e] + ([x * e, x ** 2 * e] if interaction else [])
    return np.column_stack(cols)

def fit_logit(X, y, cl=None):
    """Unpenalised logistic regression. With cluster labels cl it also returns the cluster-robust (sandwich) covariance."""
    m = LogisticRegression(C=np.inf, max_iter=5000).fit(X, y)
    th = np.r_[m.intercept_, m.coef_[0]]
    if cl is None:
        return th, None
    p = m.predict_proba(X)[:, 1]
    Xd = np.column_stack([np.ones(len(X)), X])
    Ainv = np.linalg.inv((Xd * (p * (1 - p))[:, None]).T @ Xd)                 # bread
    S = pd.DataFrame(Xd * (y - p)[:, None]).groupby(cl).sum().to_numpy()      # one score vector per tournament (meat)
    G = len(S)
    return th, Ainv @ (S.T @ S) @ Ainv * G / max(G - 1, 1)

def predict(th, X):
    return 1 / (1 + np.exp(-(np.column_stack([np.ones(len(X)), X]) @ th)))

def first_cross(p, al):
    """First grid gap where p < alpha (works on one curve or on a matrix of curves, one per row); NaN if never."""
    below = p < al
    return np.where(below.any(axis=-1), GRID[below.argmax(axis=-1)], np.nan)

def prepare(games, mode):
    """Games with an underdog for one run; adds tournament cluster, hash split and the 0/1 outcomes."""
    keep = ["game_url", "broadcast_name", "period", "tc_class", "underdog_color", "gap", "avg_elo", "underdog_score", "elo_mismatch_any", "tc_borderline",
            "white_elo_final", "black_elo_final", "white_elo_fide", "black_elo_fide"]
    g = games.loc[games["has_underdog"], keep].copy()                       # only the columns the model needs (saves memory)
    note = ""
    if mode == "sens":
        n0 = len(g)
        g = g[~g["elo_mismatch_any"].fillna(False) & ~g["tc_borderline"].fillna(False)].copy()
        note = f"SENSITIVITY RUN: dropped {n0 - len(g)} games (Elo mismatch >100 or borderline time control); {len(g)} left"
    if mode == "verified":
        n0 = len(g)
        ok = lambda s: g[f"{s}_elo_fide"].notna() & ((g[f"{s}_elo_final"] - g[f"{s}_elo_fide"]).abs() <= 100)
        g = g[ok("white") & ok("black")].copy()
        note = f"VERIFIED RUN: keeping only games where BOTH ratings are confirmed by the FIDE list (within 100 points): {len(g)} of {n0} games"
    # cluster = the tournament (broadcast_name) and year; broadcast_url would be one ROUND, so rounds of one event would land in both halves
    g["cluster"] = g["broadcast_name"].astype(str) + "|" + g["period"].astype(str).str[:4]
    g["cl_code"] = pd.factorize(g["cluster"])[0]
    g["train"] = pd.util.hash_array(g["cluster"].to_numpy(dtype=object)) % 2 == 0      # half of the tournaments
    g["win"] = (g["underdog_score"] == 1).astype(int)
    g["notlose"] = (g["underdog_score"] > 0).astype(int)
    return g, note

SUFFIX = {"main": "", "verified": "_verified", "sens": "_sens", "interaction": "_interaction"}
RUNS = {"main": "", "verified": "_verified", "sensitivity": "_sens", "interaction": "_interaction"}     # display name -> file suffix
ORDER_BAND = ["<2200", "2200-2499", "2500+"]
def load_crit(suf): return pd.read_csv(os.path.join(OUT, f"critical_gap_empirical{suf}.csv"))
def fmt(v): return "never" if pd.isna(v) else f"{v:.0f}"
def ci(r, col="critical_gap"): return f"{fmt(r[col])} ({fmt(r['lo95'])}–{fmt(r['hi95'])})"

In [8]:
def run_model(games, mode="main"):
    """One complete run. mode: main | verified | sens | interaction (all games, design with rating x gap terms).
    Writes the files <name><suffix> to OUT and the text report to reports/model_outputs/; returns a dict of results."""
    t0 = time.time()
    inter, SUF = mode == "interaction", SUFFIX[mode]
    rng = np.random.default_rng(SEED_CI)
    out = []
    def P(*a): out.append(" ".join(str(x) for x in a))
    g, note = prepare(games, "main" if inter else mode)
    if inter: note = "INTERACTION RUN: all games; design adds x*(avg_elo-2300)/100 and x^2*(avg_elo-2300)/100"
    if note: P(note)
    P(f"tournaments (clusters): {g['cl_code'].nunique()}; games in training half: {int(g.train.sum())}, in held-out half: {int((~g.train).sum())}")
    D = lambda gap, avg: design(gap, avg, inter)
    models = {}                                                              # (class, colour, outcome, train|all) -> (theta, cov)
    for c in CLASSES:
        for col in COLOURS:
            sub = g[(g.tc_class == c) & (g.underdog_color == col)]
            for which, d in (("train", sub[sub.train]), ("all", sub)):
                X = D(d.gap, d.avg_elo)
                for oc in ("win", "notlose"):
                    models[(c, col, oc, which)] = fit_logit(X, d[oc].to_numpy(), d["cl_code"].to_numpy() if which == "all" else None)
    # ---- held-out calibration
    te = g[~g.train].copy(); te["p_win"] = np.nan; te["p_nl"] = np.nan
    for c in CLASSES:
        for col in COLOURS:
            m = ((te.tc_class == c) & (te.underdog_color == col)).to_numpy()
            X = D(te.gap[m], te.avg_elo[m])
            te.loc[m, "p_win"] = predict(models[(c, col, "win", "train")][0], X)
            te.loc[m, "p_nl"] = predict(models[(c, col, "notlose", "train")][0], X)
    te["gap100"] = (np.minimum(te.gap, 700) // 100 * 100).astype(int)
    cal = (te.groupby(["tc_class", "underdog_color", "gap100"])
             .agg(n=("win", "size"), obs_win=("win", "mean"), pred_win=("p_win", "mean"),
                  obs_notlose=("notlose", "mean"), pred_notlose=("p_nl", "mean")).reset_index())
    cal.to_csv(os.path.join(OUT, f"calibration_empirical{SUF}.csv"), index=False)
    P("== held-out calibration (tournaments not used for fitting): observed vs predicted, underdog win and not-lose")
    P(cal.round(4).to_string(index=False))
    w = lambda a, b: float(np.average(np.abs(cal[a] - cal[b]), weights=cal["n"]))
    P(f"\nweighted mean |observed - predicted|: win {w('obs_win','pred_win'):.4f}   not-lose {w('obs_notlose','pred_notlose'):.4f}")
    # ---- critical gaps with intervals
    draws, rows = {}, []
    for c in CLASSES:
        for col in COLOURS:
            th, cov = models[(c, col, "win", "all")]
            samp = rng.multivariate_normal(th, cov, B_DRAWS)
            for band, e in REF_ELO.items():
                Xg = D(GRID, np.full(len(GRID), e))
                Z = np.column_stack([np.ones(len(GRID)), Xg])
                pg = 1 / (1 + np.exp(-(samp @ Z.T)))                          # B x grid
                p0 = predict(th, Xg)
                for al in ALPHAS:
                    cg = first_cross(pg, al); draws[(c, col, band, al)] = cg
                    rows.append(dict(tc_class=c, underdog_colour=col, band=band, ref_avg_elo=e, alpha=al, critical_gap=float(first_cross(p0, al)),
                                     lo95=float(np.nanpercentile(cg, 2.5)) if np.isfinite(cg).any() else np.nan,
                                     hi95=float(np.nanpercentile(cg, 97.5)) if np.isfinite(cg).any() else np.nan))
    crit = pd.DataFrame(rows)
    crit.to_csv(os.path.join(OUT, f"critical_gap_empirical{SUF}.csv"), index=False)
    drows = []
    for col in COLOURS:
        for band in REF_ELO:
            for al in ALPHAS:
                for a_, b_ in (("classical", "rapid"), ("rapid", "blitz"), ("classical", "blitz")):
                    d = draws[(a_, col, band, al)] - draws[(b_, col, band, al)]
                    d = d[np.isfinite(d)]
                    if len(d):
                        drows.append(dict(underdog_colour=col, band=band, alpha=al, pair=f"{a_} - {b_}", diff=float(np.median(d)),
                                          lo95=float(np.percentile(d, 2.5)), hi95=float(np.percentile(d, 97.5))))
    diff = pd.DataFrame(drows)
    diff.to_csv(os.path.join(OUT, f"critical_gap_diff_empirical{SUF}.csv"), index=False)
    # ---- coefficients and fitted curves (own generator, so the intervals above do not depend on this step)
    rng2 = np.random.default_rng(SEED_CURVES)
    names = ["intercept", "b_gap", "b_gap2", "b_gap3", "b_avgelo"] + (["b_gap_avgelo", "b_gap2_avgelo"] if inter else [])
    cf, fc = [], []
    for c in CLASSES:
        for col in COLOURS:
            th_w, cov_w = models[(c, col, "win", "all")]
            th_n, _ = models[(c, col, "notlose", "all")]
            for oc, th in (("win", th_w), ("notlose", th_n)):
                cf.append(dict(tc_class=c, underdog_colour=col, outcome=oc, **dict(zip(names, th))))
            samp = rng2.multivariate_normal(th_w, cov_w, 500)
            for band, e in REF_ELO.items():
                Xg = D(GRID, np.full(len(GRID), e))
                Z = np.column_stack([np.ones(len(GRID)), Xg])
                pg = 1 / (1 + np.exp(-(samp @ Z.T)))
                fc.append(pd.DataFrame(dict(tc_class=c, underdog_colour=col, band=band, ref_avg_elo=e, gap=GRID,
                          p_win=predict(th_w, Xg), p_win_lo95=np.percentile(pg, 2.5, axis=0), p_win_hi95=np.percentile(pg, 97.5, axis=0),
                          p_notlose=np.maximum(predict(th_n, Xg), predict(th_w, Xg)))))
    pd.DataFrame(cf).to_csv(os.path.join(OUT, f"model_coefficients_empirical{SUF}.csv"), index=False)
    pd.concat(fc).to_csv(os.path.join(OUT, f"fitted_curves{SUF}.csv"), index=False)
    P("\n== critical gap: smallest rating gap where P(underdog wins) < alpha  (95% interval); band = strength level of the players")
    for al in ALPHAS[1:]:
        P(f"\n  alpha = {al}")
        t = crit[crit.alpha == al].copy()
        t["gap (95% CI)"] = t.apply(lambda r: f"{r.critical_gap:.0f} ({r.lo95:.0f}-{r.hi95:.0f})", axis=1)
        P(t.pivot_table(index=["tc_class", "band"], columns="underdog_colour", values="gap (95% CI)", aggfunc="first").to_string())
    P("\n== does it change with time control? difference in critical gap (band 2200-2499, alpha=0.10); interval from the same draws")
    for col in COLOURS:
        for a_, b_ in (("classical", "rapid"), ("classical", "blitz"), ("rapid", "blitz")):
            d = draws[(a_, col, "2200-2499", 0.10)] - draws[(b_, col, "2200-2499", 0.10)]
            d = d[np.isfinite(d)]
            P(f"   {col:5s} underdog: {a_} minus {b_}: {np.median(d):+.0f}  (95% {np.percentile(d,2.5):+.0f} to {np.percentile(d,97.5):+.0f})")
    P("   (nan in the tables above = the curve does not fall below alpha within a 900-point gap; there is too little data beyond that)")
    P("   an interval that excludes 0 means the critical gap really differs between the two time controls.")
    if not inter:
        # ---- critical gap on decisive games only (draws removed): share of the non-drawn games that the underdog wins
        P("\n== critical gap on decisive games: smallest gap where the underdog wins less than alpha of the games that are NOT drawn")
        P("   (removes the effect of different draw rates; band 2200-2499, avg Elo 2350; interval from independent draws of the two fits)")
        dres = []
        for c in CLASSES:
            for col in COLOURS:
                thw, cw = models[(c, col, "win", "all")]; thn, cn = models[(c, col, "notlose", "all")]
                sw_, sn_ = rng.multivariate_normal(thw, cw, B_DRAWS), rng.multivariate_normal(thn, cn, B_DRAWS)
                Z = np.column_stack([np.ones(len(GRID)), D(GRID, np.full(len(GRID), 2350))])
                pw_, pn_ = 1 / (1 + np.exp(-(sw_ @ Z.T))), 1 / (1 + np.exp(-(sn_ @ Z.T)))
                pn_ = np.maximum(pn_, pw_)
                share = pw_ / np.clip(pw_ + 1 - pn_, 1e-9, None)
                p0w, p0n = predict(thw, D(GRID, np.full(len(GRID), 2350))), predict(thn, D(GRID, np.full(len(GRID), 2350)))
                sh0 = p0w / (p0w + 1 - np.maximum(p0n, p0w))
                for al in ALPHAS[:3]:
                    cg = first_cross(share, al); pt = float(first_cross(sh0, al))
                    dres.append(dict(tc_class=c, underdog_colour=col, alpha=al, critical_gap_decisive=pt,
                                     lo95=float(np.nanpercentile(cg, 2.5)) if np.isfinite(cg).any() else np.nan,
                                     hi95=float(np.nanpercentile(cg, 97.5)) if np.isfinite(cg).any() else np.nan))
        dres = pd.DataFrame(dres); dres.to_csv(os.path.join(OUT, f"critical_gap_decisive{SUF}.csv"), index=False)
        dres["gap (95% CI)"] = dres.apply(lambda r: f"{r.critical_gap_decisive:.0f} ({r.lo95:.0f}-{r.hi95:.0f})", axis=1)
        P(dres.pivot_table(index=["alpha", "tc_class"], columns="underdog_colour", values="gap (95% CI)", aggfunc="first").to_string())
        # ---- no-model cross-check: isotonic (monotone decreasing) curve of the observed win rate, strength levels pooled
        iso = []
        for c in CLASSES:
            for col in COLOURS:
                d = g[(g.tc_class == c) & (g.underdog_color == col)]
                ir = IsotonicRegression(increasing=False, out_of_bounds="clip").fit(np.minimum(d.gap, 1500), d["win"])
                fitv = ir.predict(GRID)
                for al in ALPHAS[:3]:
                    iso.append(dict(tc_class=c, underdog_colour=col, alpha=al,
                                    critical_gap_isotonic=float(GRID[(fitv < al).argmax()]) if (fitv < al).any() else np.nan))
        iso = pd.DataFrame(iso); iso.to_csv(os.path.join(OUT, f"critical_gap_isotonic{SUF}.csv"), index=False)
        P("\n== no-model cross-check: isotonic (monotone) curve of the observed win rate, all strength levels pooled (nan = never falls below alpha in the data)")
        P(iso.pivot_table(index=["alpha", "tc_class"], columns="underdog_colour", values="critical_gap_isotonic").to_string())
    tail = g[g.gap >= 700].groupby(["tc_class", "underdog_color"]).agg(n=("win", "size"), win=("win", "mean"), not_lose=("notlose", "mean")).round(4)
    P("\n== games with gap >= 700: how often do underdogs still win / not lose?  (tiny n means treat with care)")
    P(tail.to_string())
    # ---- final upset flags on every game
    g["p_win_emp"] = np.nan; g["p_notlose_emp"] = np.nan
    for c in CLASSES:
        for col in COLOURS:
            m = ((g.tc_class == c) & (g.underdog_color == col)).to_numpy()
            X = D(g.gap[m], g.avg_elo[m])
            g.loc[m, "p_win_emp"] = predict(models[(c, col, "win", "all")][0], X)
            g.loc[m, "p_notlose_emp"] = predict(models[(c, col, "notlose", "all")][0], X)
    g["p_notlose_emp"] = np.maximum(g["p_notlose_emp"], g["p_win_emp"])
    us = g["underdog_score"].to_numpy()
    g["q_emp"] = np.where(us == 1, g["p_win_emp"], np.where(us == 0.5, g["p_notlose_emp"], 1.0))
    for al in ALPHAS:
        g[f"upset_emp_{al:.2f}"] = (us > 0) & (g["q_emp"] < al)
    flags = g[["game_url", "p_win_emp", "p_notlose_emp", "q_emp"] + [f"upset_emp_{a:.2f}" for a in ALPHAS]]
    flags.to_parquet(os.path.join(OUT, f"upsets_empirical{SUF}.parquet"), index=False)
    P("\n== upsets under the final (empirical) definition")
    for al in ALPHAS:
        f = g[f"upset_emp_{al:.2f}"]
        P(f"alpha={al:.2f}: " + ", ".join(f"{c} {int(f[g.tc_class == c].sum())} ({f[g.tc_class == c].mean():.2%})" for c in CLASSES)
          + "   | by underdog result: wins %d, draws %d" % (int((f & (us == 1)).sum()), int((f & (us == .5)).sum())))
    mono = np.mean([np.any(np.diff(predict(models[(c, col, 'win', 'all')][0], D(GRID[GRID <= 900], np.full((GRID <= 900).sum(), 2350)))) > 1e-9)
                    for c in CLASSES for col in COLOURS])
    P("\nmonotonicity check (P(win) should fall as the gap grows): share of fitted curves with a rise before gap 900 =", f"{mono:.0%}")
    txt = "\n".join(out)
    open(os.path.join(SMALL_OUT, f"empirical_report{SUF}.txt"), "w", encoding="utf-8").write(txt + "\n")
    err = cal.assign(e=(cal.obs_win - cal.pred_win).abs())
    res = dict(mode=mode, games=len(g), crit=crit, diff=diff, cal=cal, coef=pd.DataFrame(cf), seconds=time.time() - t0,
               cal_weighted=float(np.average(err.e, weights=err.n)), cal_unweighted=float(err.e.mean()), cal_max=float(err.e.max()),
               upsets={al: int(g[f"upset_emp_{al:.2f}"].sum()) for al in ALPHAS})
    return res

def summary(res):
    say(f"**Run `{res['mode']}`**: {res['games']:,} games, 6 slices × 2 outcomes fitted twice (training half, all games) in {res['seconds']:.0f} s. "
        f"Held-out calibration error of P(win): **{100 * res['cal_weighted']:.2f}** percentage points (n-weighted), {100 * res['cal_unweighted']:.2f} (unweighted), "
        f"largest {100 * res['cal_max']:.2f}. Upsets at α = 0.20 / 0.10 / 0.05 / 0.01: " + " / ".join(f"{v:,}" for v in res["upsets"].values()) + ".")

### 3.7 A worked case

Classical games with White as the underdog: the fit of P(win), the naive and the clustered standard errors of the coefficients, the fitted probabilities at a few gaps for players averaging 2350, and how the upset flag works for two typical results.

In [9]:
gx, _ = prepare(games, "main")
d = gx[(gx.tc_class == "classical") & (gx.underdog_color == "white")]
X = design(d.gap, d.avg_elo)
th, V = fit_logit(X, d["win"].to_numpy(), d["cl_code"].to_numpy())
p = predict(th, X); Xd = np.column_stack([np.ones(len(X)), X])
naive = np.sqrt(np.diag(np.linalg.inv((Xd * (p * (1 - p))[:, None]).T @ Xd)))
coef = pd.DataFrame({"coefficient": th, "naive_se": naive, "clustered_se": np.sqrt(np.diag(V))},
                    index=["b0 intercept", "b1 x", "b2 x^2", "b3 x^3", "b4 e (rating)"])
coef["ratio"] = coef.clustered_se / coef.naive_se
display(coef.round(4))
gaps = np.array([0, 100, 200, 300, 400, 500, 600])
pw = predict(th, design(gaps, np.full(len(gaps), 2350)))
thn, _ = fit_logit(X, d["notlose"].to_numpy())
pn = np.maximum(predict(thn, design(gaps, np.full(len(gaps), 2350))), pw)
display(pd.DataFrame({"gap": gaps, "P(win)": pw.round(3), "P(not lose)": pn.round(3)}).set_index("gap").T)
cg = float(first_cross(predict(th, design(GRID, np.full(len(GRID), 2350))), 0.10))
lev = lambda q: ("an upset at α = " + " and ".join(f"{a:.2f}" for a in ALPHAS if q < a)) if q < max(ALPHAS) else "not an upset at any α used here"
say(f"{len(d):,} games from {d.cl_code.nunique():,} tournaments. The clustered standard errors are **{coef.ratio.min():.1f}–{coef.ratio.max():.1f} times** the naive ones: "
    f"treating the games as independent would understate the uncertainty. For players averaging 2350 the fitted P(win) first drops below 0.10 at a gap of **{cg:.0f}** points: "
    f"that is the critical gap of this slice. Example: at a gap of 300 a White underdog **wins** with probability {pw[3]:.3f} (a win there has q = {pw[3]:.3f}: {lev(pw[3])}), "
    f"and **draws or wins** with probability {pn[3]:.3f} (a draw has q = {pn[3]:.3f}: {lev(pn[3])}).")
del gx, d, X, p, Xd

,coefficient,naive_se,clustered_se,ratio
b0 intercept,-0.9042,0.0133,0.0142,1.0647
b1 x,-1.3426,0.0645,0.0685,1.0620
b2 x^2,-0.0601,0.0842,0.0905,1.0747
b3 x^3,0.0372,0.0294,0.0322,1.0971
b4 e (rating),-0.0797,0.0015,0.0019,1.2433


gap,0,100,200,300,400,500,600
P(win),0.280,0.198,0.135,0.090,0.06,0.040,0.027
P(not lose),0.784,0.666,0.527,0.392,0.28,0.198,0.144


303,554 games from 7,475 tournaments. The clustered standard errors are **1.1–1.2 times** the naive ones: treating the games as independent would understate the uncertainty. For players averaging 2350 the fitted P(win) first drops below 0.10 at a gap of **280** points: that is the critical gap of this slice. Example: at a gap of 300 a White underdog **wins** with probability 0.090 (a win there has q = 0.090: an upset at α = 0.20 and 0.10), and **draws or wins** with probability 0.392 (a draw has q = 0.392: not an upset at any α used here).

## 4. Fitting the model: 3 time controls × 2 underdog colours

The main run uses all games with an underdog. For each slice it fits P(win) and P(not lose) twice: on the training half of the tournaments (for section 7) and on all games (for everything else).

In [10]:
main = run_model(games, "main")
summary(main)

**Run `main`**: 862,599 games, 6 slices × 2 outcomes fitted twice (training half, all games) in 5 s. Held-out calibration error of P(win): **0.39** percentage points (n-weighted), 0.72 (unweighted), largest 2.23. Upsets at α = 0.20 / 0.10 / 0.05 / 0.01: 71,079 / 11,114 / 1,183 / 0.

In [11]:
coef = pd.read_csv(os.path.join(OUT, "model_coefficients_empirical.csv"))
display(coef.set_index(["tc_class", "underdog_colour", "outcome"]).round(3))
bw, bn = coef[coef.outcome == "win"].b_avgelo, coef[coef.outcome == "notlose"].b_avgelo
say("The coefficients of a cubic are hard to read one by one; the curves they describe (sections 5 and 9) are what matter. "
    f"The rating term b_avgelo is {'negative' if (bw < 0).all() else 'of mixed sign'} in all six P(win) fits ({bw.min():.3f} to {bw.max():.3f}): at the same gap, the underdog wins less often among stronger players. "
    f"In the P(not lose) fits it is {'positive' if (bn > 0).all() else 'of mixed sign'} ({bn.min():.3f} to {bn.max():.3f}): stronger players draw more.")

intercept  b_gap  b_gap2  b_gap3  b_avgelo
tc_class  underdog_colour outcome                                            
classical white           win         -0.904 -1.343  -0.060   0.037    -0.080
                          notlose      1.254 -1.809   0.027   0.055     0.064
          black           win         -1.385 -1.034  -0.490   0.178    -0.106
                          notlose      0.780 -1.925   0.161   0.023     0.029
rapid     white           win         -0.554 -1.008  -0.317   0.121    -0.034
                          notlose      0.899 -1.442  -0.083   0.077     0.066
          black           win         -0.873 -1.127  -0.244   0.116    -0.056
                          notlose      0.523 -1.498  -0.078   0.102     0.038
blitz     white           win         -0.386 -0.938  -0.272   0.082    -0.030
                          notlose      0.582 -1.330   0.010   0.013     0.037
          black           win         -0.628 -0.795  -0.485   0.161    -0.043
                          notlose      0.311 -1.281  -0.067   0.050     0.024

The coefficients of a cubic are hard to read one by one; the curves they describe (sections 5 and 9) are what matter. The rating term b_avgelo is negative in all six P(win) fits (-0.106 to -0.030): at the same gap, the underdog wins less often among stronger players. In the P(not lose) fits it is positive (0.024 to 0.066): stronger players draw more.

## 5. The critical gaps

Critical gap (rating points) with its 95 % interval, for the three strength bands (reference average rating 2100 / 2350 / 2600), both underdog colours and the four thresholds α. "never" = the fitted P(win) stays above α at every gap.

In [12]:
crit = pd.read_csv(os.path.join(OUT, "critical_gap_empirical.csv"))
crit["gap (95% interval)"] = crit.apply(ci, axis=1)
for al in ALPHAS:
    t = crit[crit.alpha == al].pivot_table(index=["tc_class", "band"], columns="underdog_colour", values="gap (95% interval)", aggfunc="first")
    t = t.reindex(pd.MultiIndex.from_product([CLASSES, ORDER_BAND]))[["white", "black"]]
    say(f"**α = {al:.2f}**"); display(t)
r = crit.set_index(["tc_class", "underdog_colour", "band", "alpha"])
k = lambda c, col: r.loc[(c, col, "2200-2499", 0.10)]
say("**Headline (α = 0.10, players averaging 2350).** An underdog's **win** becomes an upset beyond a gap of "
    + ", ".join(f"**{fmt(k(c, 'white').critical_gap)}** ({fmt(k(c, 'white').lo95)}–{fmt(k(c, 'white').hi95)}) in {c}" for c in CLASSES)
    + " when the underdog has White; with Black: " + ", ".join(f"**{fmt(k(c, 'black').critical_gap)}** in {c}" for c in CLASSES)
    + ". The faster the game, the larger the gap needed: in blitz an underdog's win is far less surprising.")

**α = 0.20**

underdog_colour              white          black
classical <2200      145 (140–145)     60 (55–65)
          2200-2499   100 (95–105)        0 (0–0)
          2500+         55 (50–60)        0 (0–0)
rapid     <2200      230 (225–240)  155 (150–165)
          2200-2499  210 (200–220)  125 (115–130)
          2500+      190 (180–205)    90 (80–100)
blitz     <2200      285 (275–300)  240 (225–250)
          2200-2499  270 (255–280)  210 (200–225)
          2500+      250 (235–265)  185 (170–200)

**α = 0.10**

underdog_colour              white          black
classical <2200      320 (315–330)  235 (235–240)
          2200-2499  280 (270–285)  185 (180–185)
          2500+      235 (225–240)  125 (120–130)
rapid     <2200      425 (405–450)  345 (335–360)
          2200-2499  405 (385–425)  310 (300–325)
          2500+      385 (365–405)  280 (265–295)
blitz     <2200      485 (460–515)  430 (405–455)
          2200-2499  465 (440–495)  405 (380–430)
          2500+      450 (425–475)  380 (355–405)

**α = 0.05**

underdog_colour              white          black
classical <2200      495 (480–510)  390 (380–400)
          2200-2499  445 (435–460)  335 (330–345)
          2500+      400 (390–410)  280 (275–290)
rapid     <2200      660 (600–745)  560 (520–620)
          2200-2499  625 (575–695)  510 (480–555)
          2500+      595 (550–650)  470 (440–505)
blitz     <2200      700 (635–795)  670 (600–745)
          2200-2499  675 (616–775)  620 (570–699)
          2500+      650 (595–750)  580 (535–650)

**α = 0.01**

underdog_colour                    white                black
classical <2200          never (816–900)      never (855–860)
          2200-2499      never (785–900)      never (744–850)
          2500+            855 (755–895)      never (655–774)
rapid     <2200      never (never–never)  never (never–never)
          2200-2499  never (never–never)  never (never–never)
          2500+      never (never–never)  never (never–never)
blitz     <2200      never (never–never)  never (never–never)
          2200-2499  never (never–never)  never (never–never)
          2500+      never (never–never)  never (never–never)

**Headline (α = 0.10, players averaging 2350).** An underdog's **win** becomes an upset beyond a gap of **280** (270–285) in classical, **405** (385–425) in rapid, **465** (440–495) in blitz when the underdog has White; with Black: **185** in classical, **310** in rapid, **405** in blitz. The faster the game, the larger the gap needed: in blitz an underdog's win is far less surprising.

## 6. Do the critical gaps differ between time controls?

The difference between the critical gaps of two time controls, computed for each of the 1,000 parameter draws (median and 2.5 / 97.5 % points). The median of the differences need not equal the difference of the two point estimates exactly (both live on a 5-point grid). An interval that excludes 0 means the critical gap really differs.

In [13]:
diff = pd.read_csv(os.path.join(OUT, "critical_gap_diff_empirical.csv"))
diff["difference (95% interval)"] = diff.apply(lambda r: f"{r['diff']:+.0f} ({r.lo95:+.0f} to {r.hi95:+.0f})", axis=1)
diff["excludes 0"] = (diff.hi95 < 0) | (diff.lo95 > 0)
for al in (0.20, 0.10, 0.05):
    t = diff[diff.alpha == al].pivot_table(index=["band", "pair"], columns="underdog_colour", values="difference (95% interval)", aggfunc="first")
    say(f"**α = {al:.2f}**"); display(t.reindex(ORDER_BAND, level=0)[["white", "black"]])
h = diff[diff.alpha.isin([0.20, 0.10, 0.05])]
say(f"At α = 0.20 / 0.10 / 0.05, **{int(h['excludes 0'].sum())} of {len(h)}** differences have an interval that excludes 0. "
    "Those that do not: " + (", ".join(f"{r.pair} ({r.underdog_colour}, {r.band}, α = {r.alpha:.2f})" for r in h[~h['excludes 0']].itertuples()) or "none") + ".")

**α = 0.20**

underdog_colour                            white                black
band      pair                                                       
<2200     classical - blitz  -140 (-155 to -130)  -180 (-190 to -165)
          classical - rapid     -85 (-95 to -80)    -95 (-105 to -85)
          rapid - blitz         -55 (-70 to -40)     -80 (-95 to -70)
2200-2499 classical - blitz  -170 (-180 to -155)  -210 (-225 to -200)
          classical - rapid  -110 (-125 to -100)  -125 (-130 to -115)
          rapid - blitz         -55 (-70 to -40)    -90 (-105 to -75)
2500+     classical - blitz  -195 (-210 to -175)  -185 (-200 to -170)
          classical - rapid  -135 (-150 to -120)    -90 (-100 to -80)
          rapid - blitz         -60 (-80 to -35)    -95 (-115 to -80)

**α = 0.10**

underdog_colour                            white                black
band      pair                                                       
<2200     classical - blitz  -165 (-195 to -135)  -190 (-220 to -170)
          classical - rapid   -105 (-125 to -85)   -110 (-125 to -95)
          rapid - blitz         -60 (-95 to -25)    -85 (-115 to -55)
2200-2499 classical - blitz  -190 (-220 to -160)  -220 (-245 to -200)
          classical - rapid  -125 (-150 to -105)  -130 (-145 to -115)
          rapid - blitz         -60 (-95 to -30)    -90 (-120 to -65)
2500+     classical - blitz  -215 (-245 to -190)  -250 (-275 to -230)
          classical - rapid  -150 (-170 to -130)  -155 (-170 to -140)
          rapid - blitz        -65 (-100 to -35)   -100 (-125 to -70)

**α = 0.05**

underdog_colour                            white                black
band      pair                                                       
<2200     classical - blitz  -200 (-300 to -140)  -270 (-355 to -210)
          classical - rapid  -160 (-250 to -105)  -165 (-230 to -125)
          rapid - blitz        -40 (-145 to +65)   -100 (-195 to -15)
2200-2499 classical - blitz  -225 (-324 to -166)  -285 (-365 to -231)
          classical - rapid  -175 (-246 to -125)  -175 (-220 to -145)
          rapid - blitz        -50 (-155 to +45)   -105 (-195 to -40)
2500+     classical - blitz  -250 (-347 to -195)  -300 (-370 to -255)
          classical - rapid  -190 (-250 to -150)  -190 (-220 to -160)
          rapid - blitz        -60 (-152 to +20)   -110 (-185 to -55)

At α = 0.20 / 0.10 / 0.05, **51 of 54** differences have an interval that excludes 0. Those that do not: rapid - blitz (white, <2200, α = 0.05), rapid - blitz (white, 2200-2499, α = 0.05), rapid - blitz (white, 2500+, α = 0.05).

## 7. Does the model predict tournaments it has never seen?

The model fitted on one half of the tournaments predicts the other half. For each slice and 100-point gap bin (700 = 700 and more) the table compares the observed share of underdog wins (and wins or draws) with the mean predicted probability. The error is |observed − predicted|, averaged over the 48 bins, once weighted by the number of games and once unweighted (each bin counts the same, so the sparse large-gap bins weigh more).

In [14]:
cal = pd.read_csv(os.path.join(OUT, "calibration_empirical.csv"))
cal["err_win"], cal["err_notlose"] = (cal.obs_win - cal.pred_win).abs(), (cal.obs_notlose - cal.pred_notlose).abs()
t = cal.pivot_table(index="gap100", columns=["tc_class", "underdog_color"], values=["obs_win", "pred_win"])
display((100 * t).round(1).reindex(columns=list(CLASSES), level=1))
def errs(d):
    return pd.Series({"bins": len(d), "games": int(d.n.sum()),
                      "P(win) weighted": 100 * np.average(d.err_win, weights=d.n), "P(win) unweighted": 100 * d.err_win.mean(), "P(win) max": 100 * d.err_win.max(),
                      "P(not lose) weighted": 100 * np.average(d.err_notlose, weights=d.n), "P(not lose) unweighted": 100 * d.err_notlose.mean()})
et = pd.concat([cal.groupby("tc_class").apply(errs, include_groups=False).reindex(list(CLASSES)), errs(cal).to_frame("all").T])
display(et.round(2))
wst = cal.loc[cal.err_win.idxmax()]
mid = cal[cal.tc_class.isin(["rapid", "blitz"]) & cal.gap100.between(200, 500)]
say(f"Mean absolute error of P(win) on unseen tournaments: **{et.loc['all', 'P(win) weighted']:.2f} percentage points** (n-weighted), "
    f"**{et.loc['all', 'P(win) unweighted']:.2f}** (unweighted); the largest error is {et.loc['all', 'P(win) max']:.2f} points "
    f"({wst.tc_class}, {wst.underdog_color} underdog, gap {wst.gap100:.0f}–{wst.gap100 + 99:.0f}, n = {wst.n:,.0f}). "
    f"In rapid and blitz at gaps 200–599 the observed win share is below the prediction in {int((mid.obs_win < mid.pred_win).sum())} of {len(mid)} bins "
    f"(mean {100 * (mid.obs_win - mid.pred_win).mean():+.1f} points): there the model is slightly generous to the underdog.")

obs_win                                pred_win                              
tc_class       classical       rapid       blitz       classical       rapid       blitz      
underdog_color     black white black white black white     black white black white black white
gap100                                                                                        
0                   20.8  26.9  27.3  33.6  31.8  36.2      20.4  27.5  28.0  34.2  31.8  36.8
100                 15.1  20.3  20.7  26.2  25.5  30.0      15.1  20.1  21.3  26.4  25.7  29.7
200                 10.3  14.3  14.9  19.1  18.3  21.8      10.3  14.1  15.7  19.8  19.8  23.1
300                  7.3  10.1   9.8  12.8  14.2  15.9       6.8   9.6  10.9  14.4  14.2  17.0
400                  4.8   6.6   7.3   8.6   9.2  11.1       4.4   6.5   7.6  10.5  10.1  12.4
500                  3.3   5.3   5.9   5.9   6.4   6.9       3.0   4.5   5.5   7.8   7.5   8.9
600                  2.2   2.9   4.6   5.6   3.8   5.3       2.3   3.2   4.1   6.0   6.0   6.5
700                  2.3   2.3   4.9   3.8   3.9   4.4       2.1   2.2   3.3   4.7   5.8   4.5

,bins,games,P(win) weighted,P(win) unweighted,P(win) max,P(not lose) weighted,P(not lose) unweighted
classical,16.0,301078.0,0.26,0.29,0.83,0.44,0.93
rapid,16.0,72136.0,0.74,0.90,1.95,0.94,1.61
blitz,16.0,53478.0,0.63,0.98,2.23,0.90,1.51
all,48.0,426692.0,0.39,0.72,2.23,0.58,1.35


Mean absolute error of P(win) on unseen tournaments: **0.39 percentage points** (n-weighted), **0.72** (unweighted); the largest error is 2.23 points (blitz, black underdog, gap 600–699, n = 373). In rapid and blitz at gaps 200–599 the observed win share is below the prediction in 15 of 16 bins (mean -1.1 points): there the model is slightly generous to the underdog.

## 8. Robustness: do the conclusions survive different choices?

| Run | What changes | Files |
|---|---|---|
| main | all games with an underdog | no suffix |
| verified | only games whose two PGN ratings are both confirmed by the FIDE list within 100 points (ratings that can be trusted) | `_verified` |
| sensitivity | drops games whose PGN rating differs from FIDE by more than 100 points and games whose time control lies within 3 minutes of a class boundary | `_sens` |
| isotonic | no logistic model at all: a monotone decreasing curve through the observed win rate (strength levels pooled), computed in every run | `critical_gap_isotonic*` |
| decisive games | the underdog's share of the games that are *not* drawn (removes the effect of different draw rates) | `critical_gap_decisive*` |
| interaction | all games; the model gets two more terms, x·e and x²·e, so that the curve's *shape* (not only its level) can change with the players' strength | `_interaction` |

### 8.1 Verified ratings

In [15]:
ver = run_model(games, "verified")
summary(ver)

**Run `verified`**: 656,518 games, 6 slices × 2 outcomes fitted twice (training half, all games) in 4 s. Held-out calibration error of P(win): **0.37** percentage points (n-weighted), 0.81 (unweighted), largest 4.07. Upsets at α = 0.20 / 0.10 / 0.05 / 0.01: 53,408 / 8,892 / 1,231 / 1.

### 8.2 Sensitivity run

In [16]:
sens = run_model(games, "sens")
summary(sens)

**Run `sens`**: 814,370 games, 6 slices × 2 outcomes fitted twice (training half, all games) in 4 s. Held-out calibration error of P(win): **0.38** percentage points (n-weighted), 0.71 (unweighted), largest 3.62. Upsets at α = 0.20 / 0.10 / 0.05 / 0.01: 68,296 / 10,731 / 1,151 / 0.

### 8.3 Interaction model

The main model has one straight-line rating term: the strength of the players moves the whole curve up or down but cannot change how fast it falls with the gap. Section 9 shows that this makes the model's tail too generous for strong players. The interaction model adds x·e and x²·e (e = (average rating − 2300)/100) to the design, everything else unchanged (same slices, same clustered intervals, same seed). It is a robustness row: the headline numbers stay those of the main model.

In [17]:
inter = run_model(games, "interaction")
summary(inter)
display(pd.read_csv(os.path.join(OUT, "model_coefficients_empirical_interaction.csv")).set_index(["tc_class", "underdog_colour", "outcome"]).round(3))

**Run `interaction`**: 862,599 games, 6 slices × 2 outcomes fitted twice (training half, all games) in 5 s. Held-out calibration error of P(win): **0.38** percentage points (n-weighted), 0.72 (unweighted), largest 2.04. Upsets at α = 0.20 / 0.10 / 0.05 / 0.01: 70,202 / 10,776 / 1,414 / 4.

intercept  b_gap  b_gap2  b_gap3  b_avgelo  b_gap_avgelo  b_gap2_avgelo
tc_class  underdog_colour outcome                                                                         
classical white           win         -0.956 -0.976  -0.491   0.099    -0.083         0.044         -0.054
                          notlose      1.340 -1.950   0.006   0.018     0.105        -0.068         -0.019
          black           win         -1.389 -0.965  -0.490   0.070    -0.113         0.065         -0.073
                          notlose      0.847 -2.014   0.092  -0.005     0.068        -0.067         -0.020
rapid     white           win         -0.585 -0.861  -0.373   0.075    -0.049         0.076         -0.057
                          notlose      0.916 -1.420  -0.121   0.043     0.072         0.022         -0.041
          black           win         -0.950 -0.760  -0.465   0.076    -0.080         0.119         -0.087
                          notlose      0.529 -1.451  -0.124   0.068     0.039         0.032         -0.043
blitz     white           win         -0.397 -0.881  -0.273   0.030    -0.040         0.065         -0.056
                          notlose      0.585 -1.300  -0.023  -0.017     0.043         0.020         -0.040
          black           win         -0.640 -0.750  -0.469   0.118    -0.057         0.065         -0.047
                          notlose      0.308 -1.246  -0.108   0.037     0.027         0.015         -0.028

### 8.4 All checks side by side (α = 0.10)

In [18]:
rows = []
for name, suf in RUNS.items():
    c = load_crit(suf); c = c[c.alpha == 0.10]
    for x in c.itertuples():
        rows.append(dict(check=name, tc_class=x.tc_class, colour=x.underdog_colour, band=x.band, value=ci(x._asdict())))
for name, suf in (("isotonic (bands pooled)", ""), ("isotonic, verified", "_verified"), ("isotonic, sensitivity", "_sens")):
    for x in pd.read_csv(os.path.join(OUT, f"critical_gap_isotonic{suf}.csv")).query("alpha == 0.10").itertuples():
        rows.append(dict(check=name, tc_class=x.tc_class, colour=x.underdog_colour, band="pooled", value=fmt(x.critical_gap_isotonic)))
for name, suf in (("decisive games", ""), ("decisive, verified", "_verified")):
    dz = pd.read_csv(os.path.join(OUT, f"critical_gap_decisive{suf}.csv")).query("alpha == 0.10")
    for x in dz.itertuples():
        rows.append(dict(check=name, tc_class=x.tc_class, colour=x.underdog_colour, band="2200-2499", value=ci(x._asdict(), "critical_gap_decisive")))
rb = pd.DataFrame(rows)
mid = rb[rb.band.isin(["2200-2499", "pooled"])].pivot_table(index="check", columns=["colour", "tc_class"], values="value", aggfunc="first")
mid = mid.reindex(columns=pd.MultiIndex.from_product([["white", "black"], CLASSES])).reindex(list(RUNS) + ["isotonic (bands pooled)", "isotonic, verified", "isotonic, sensitivity", "decisive games", "decisive, verified"])
say("**Middle band (2200–2499; the isotonic curve pools all bands), α = 0.10**"); display(mid)
cm, ci_ = load_crit(""), load_crit("_interaction")
cmp_ = cm.merge(ci_, on=["tc_class", "underdog_colour", "band", "alpha"], suffixes=("_main", "_inter"))
cmp_["change"] = cmp_.critical_gap_inter - cmp_.critical_gap_main
cmp_["inside_main_interval"] = cmp_.critical_gap_inter.between(cmp_.lo95_main, cmp_.hi95_main)
t = cmp_[cmp_.alpha == 0.10].assign(main=lambda d: d.apply(lambda r: f"{fmt(r.critical_gap_main)} ({fmt(r.lo95_main)}–{fmt(r.hi95_main)})", axis=1),
                                     interaction=lambda d: d.apply(lambda r: f"{fmt(r.critical_gap_inter)} ({fmt(r.lo95_inter)}–{fmt(r.hi95_inter)})", axis=1))
t = t.set_index(["tc_class", "band", "underdog_colour"])[["main", "interaction", "change", "inside_main_interval"]]
say("**Main vs interaction model, all bands, α = 0.10** (change = interaction − main, rating points)")
display(t.reindex(pd.MultiIndex.from_product([CLASSES, ORDER_BAND, COLOURS])))
cmp_.to_csv(os.path.join(OUT, "critical_gap_main_vs_interaction.csv"), index=False)
ch = cmp_[cmp_.alpha == 0.10].groupby("band").change.agg(["min", "max", "mean"]).reindex(ORDER_BAND)
say("Change of the α = 0.10 critical gaps with the interaction model, by band: " + "; ".join(f"{b}: {r['min']:+.0f} to {r['max']:+.0f} (mean {r['mean']:+.0f})" for b, r in ch.iterrows())
    + f". {int(cmp_[cmp_.alpha == 0.10].inside_main_interval.sum())} of {int((cmp_.alpha == 0.10).sum())} interaction estimates lie inside the main model's 95 % interval.")
oth = cmp_[cmp_.alpha.isin([0.20, 0.05])].groupby("alpha").agg(lo=("change", "min"), hi=("change", "max"), mean=("change", "mean"), inside=("inside_main_interval", "sum"), n=("change", "size"))
say("Other thresholds: " + "; ".join(f"α = {a:.2f}: change {r.lo:+.0f} to {r.hi:+.0f} (mean {r['mean']:+.0f}), {int(r.inside)} of {int(r.n)} inside the main interval" for a, r in oth.iterrows())
    + ". The further into the tail (smaller α), the more the two models disagree: there the main model's single rating term matters most (section 9).")

**Middle band (2200–2499; the isotonic curve pools all bands), α = 0.10**

white                                        black                              
                             classical          rapid          blitz      classical          rapid          blitz
check                                                                                                            
main                     280 (270–285)  405 (385–425)  465 (440–495)  185 (180–185)  310 (300–325)  405 (380–430)
verified                 270 (265–275)  370 (350–385)  420 (405–440)  180 (175–185)  295 (280–310)  385 (360–410)
sensitivity              280 (270–285)  400 (380–420)  450 (425–480)  185 (180–185)  315 (300–330)  415 (390–445)
interaction              270 (265–280)  395 (370–420)  440 (415–470)  185 (180–190)  315 (300–335)  395 (375–430)
isotonic (bands pooled)            345            420            450            265            380            445
isotonic, verified                 315            415            450            240            325            390
isotonic, sensitivity              345            420            460            265            380            425
decisive games           350 (345–355)  450 (435–475)  495 (470–520)  250 (250–255)  355 (345–370)  430 (410–455)
decisive, verified       335 (330–340)  405 (390–425)  445 (430–465)  245 (240–250)  330 (320–345)  405 (385–430)

**Main vs interaction model, all bands, α = 0.10** (change = interaction − main, rating points)

main    interaction  change  inside_main_interval
classical <2200     white  320 (315–330)  315 (310–320)    -5.0                  True
                    black  235 (235–240)  235 (235–240)     0.0                  True
          2200-2499 white  280 (270–285)  270 (265–280)   -10.0                  True
                    black  185 (180–185)  185 (180–190)     0.0                  True
          2500+     white  235 (225–240)  235 (225–240)     0.0                  True
                    black  125 (120–130)  135 (130–140)    10.0                 False
rapid     <2200     white  425 (405–450)  420 (400–440)    -5.0                  True
                    black  345 (335–360)  345 (335–360)     0.0                  True
          2200-2499 white  405 (385–425)  395 (370–420)   -10.0                  True
                    black  310 (300–325)  315 (300–335)     5.0                  True
          2500+     white  385 (365–405)  370 (350–400)   -15.0                  True
                    black  280 (265–295)  295 (275–315)    15.0                  True
blitz     <2200     white  485 (460–515)  475 (450–510)   -10.0                  True
                    black  430 (405–455)  430 (410–460)     0.0                  True
          2200-2499 white  465 (440–495)  440 (415–470)   -25.0                  True
                    black  405 (380–430)  395 (375–430)   -10.0                  True
          2500+     white  450 (425–475)  410 (385–450)   -40.0                 False
                    black  380 (355–405)  370 (345–405)   -10.0                  True

Change of the α = 0.10 critical gaps with the interaction model, by band: <2200: -10 to +0 (mean -3); 2200-2499: -25 to +5 (mean -8); 2500+: -40 to +15 (mean -7). 16 of 18 interaction estimates lie inside the main model's 95 % interval.

Other thresholds: α = 0.05: change -120 to -10 (mean -54), 4 of 18 inside the main interval; α = 0.20: change -5 to +15 (mean +4), 16 of 18 inside the main interval. The further into the tail (smaller α), the more the two models disagree: there the main model's single rating term matters most (section 9).

## 9. Expected vs observed upsets, by strength level

If the model is right, the number of underdog wins among the games to which it gives P(win) < α should equal the sum of those games' P(win) (and the same for draws with P(not lose) < α). The ratio observed / expected should then be close to 1 in every strength band; a ratio below 1 means the model's tail is too generous (it expects more underdog wins than happen). The second table compares the model with the data in 50-point gap bins at each game's own rating: the share of bins (with ≥ 100 games) in which the model lies inside the 95 % Wilson interval of the observed share, and the mean error observed − predicted (unweighted over the bins, and weighted by their games).

In [19]:
base = games.loc[games.has_underdog, ["game_url", "tc_class", "band", "gap", "avg_elo", "underdog_color", "underdog_score"]]
def load_flags(suf):
    f = pd.read_parquet(os.path.join(OUT, f"upsets_empirical{suf}.parquet"))
    d = base.merge(f, on="game_url", how="inner", validate="one_to_one")
    d["win"], d["draw"] = (d.underdog_score == 1).astype(int), (d.underdog_score == 0.5).astype(int)
    return d
def expected_vs_observed(d0):
    rows = []
    for tc in list(CLASSES) + ["all"]:
        for band in ["all"] + ORDER_BAND:
            d = d0 if tc == "all" else d0[d0.tc_class == tc]
            d = d if band == "all" else d[d.band == band]
            for al in ALPHAS:
                bw, bn = d.p_win_emp < al, d.p_notlose_emp < al
                pdraw = (d.p_notlose_emp - d.p_win_emp).clip(lower=0)
                ew, ed = float(d.p_win_emp[bw].sum()), float(pdraw[bn].sum())
                rows.append(dict(tc_class=tc, band=band, alpha=al, games_pwin_below=int(bw.sum()), exp_wins=ew, obs_wins=int(d.win[bw].sum()),
                                 sd_wins=float(np.sqrt((d.p_win_emp[bw] * (1 - d.p_win_emp[bw])).sum())),
                                 games_pnl_below=int(bn.sum()), exp_draws=ed, obs_draws=int(d.draw[bn].sum()),
                                 exp_total=ew + ed, obs_total=int(d.win[bw].sum() + d.draw[bn].sum()), obs_flags=int(d[f"upset_emp_{al:.2f}"].sum())))
    return pd.DataFrame(rows)
def fit_by_band(d0):
    d0 = d0.assign(gap_bin50=(np.minimum(d0.gap, 900) // 50 * 50).astype(int))
    mb = d0.groupby(["tc_class", "underdog_color", "band", "gap_bin50"]).agg(n=("win", "size"), wins=("win", "sum"), pred_win=("p_win_emp", "mean")).reset_index()
    z = 1.959964; ph = mb.wins / mb.n; den = 1 + z ** 2 / mb.n
    cen = (ph + z ** 2 / (2 * mb.n)) / den; hw = z * np.sqrt(ph * (1 - ph) / mb.n + z ** 2 / (4 * mb.n ** 2)) / den
    mb["obs_win"], mb["inside"] = ph, mb.pred_win.between(cen - hw, cen + hw)
    mb = mb[mb.n >= 100]
    return mb.groupby(["tc_class", "band"]).apply(lambda d: pd.Series({"bins": len(d), "inside": int(d.inside.sum()),
                                                  "mean_err_pp": 100 * (d.obs_win - d.pred_win).mean(),
                                                  "mean_err_wtd_pp": 100 * np.average(d.obs_win - d.pred_win, weights=d.n)}), include_groups=False), mb
eo, fb = {}, {}
for name, suf in (("main", ""), ("interaction", "_interaction")):
    d = load_flags(suf)
    eo[name] = expected_vs_observed(d); fb[name], _ = fit_by_band(d)
    eo[name].round(2).to_csv(os.path.join(OUT, f"expected_upsets{suf}.csv"), index=False)
    del d
pd.concat({k: v for k, v in fb.items()}, names=["model"]).reset_index().to_csv(os.path.join(OUT, "model_fit_by_band.csv"), index=False)
both = pd.concat({k: v for k, v in eo.items()}, names=["model"]).reset_index(level=0)
both["obs/exp wins"] = both.obs_wins / both.exp_wins
t = both[both.alpha.isin([0.20, 0.10, 0.05])].pivot_table(index=["tc_class", "band"], columns=["alpha", "model"], values="obs/exp wins")
t = t.reindex(pd.MultiIndex.from_product([list(CLASSES) + ["all"], ["all"] + ORDER_BAND])).sort_index(axis=1, level=0, ascending=False)
say("**Observed / expected underdog wins among games with P(win) < α**"); display(t.round(2))
fbt = pd.concat(fb, axis=1).reindex(pd.MultiIndex.from_product([CLASSES, ORDER_BAND]))
say("**Model vs data in 50-point bins (n ≥ 100), at each game's own rating**"); display(fbt.round(2))
tot = both[(both.tc_class == "all") & (both.band == "all")].set_index(["model", "alpha"])[["exp_total", "obs_total", "obs_flags"]]
display(tot.round(0))
r10 = both[(both.tc_class == "all") & (both.alpha == 0.10)].set_index(["model", "band"])["obs/exp wins"]
say("Among games with P(win) < 0.10, observed/expected wins by band (<2200 / 2200–2499 / 2500+): main model "
    + " / ".join(f"{r10[('main', b)]:.2f}" for b in ORDER_BAND) + ", interaction model " + " / ".join(f"{r10[('interaction', b)]:.2f}" for b in ORDER_BAND)
    + f". Bins inside the 95 % interval: main {int(fb['main'].inside.sum())}/{int(fb['main'].bins.sum())}, interaction {int(fb['interaction'].inside.sum())}/{int(fb['interaction'].bins.sum())}; "
    + f"classical 2500+ n-weighted mean error {fb['main'].loc[('classical', '2500+'), 'mean_err_wtd_pp']:+.1f} (main) vs {fb['interaction'].loc[('classical', '2500+'), 'mean_err_wtd_pp']:+.1f} points (interaction).")
del base

**Observed / expected underdog wins among games with P(win) < α**

alpha                0.20              0.10              0.05            
model                main interaction  main interaction  main interaction
classical all        1.00        1.00  0.98        1.00  0.87        0.99
          <2200      1.04        1.04  1.06        1.06  0.98        1.06
          2200-2499  0.99        0.98  0.92        0.97  0.69        0.89
          2500+      0.84        0.83  0.75        0.76  0.94        1.06
rapid     all        1.00        0.99  0.99        1.01  0.86        0.98
          <2200      1.00        1.00  1.00        0.99  0.88        0.91
          2200-2499  1.01        1.00  0.91        1.06  0.76        1.15
          2500+      1.00        0.96  1.31        1.30  0.00        0.83
blitz     all        1.01        1.01  0.95        0.96  0.86        1.00
          <2200      1.04        1.02  1.04        0.98  0.90        0.92
          2200-2499  0.98        1.00  0.77        0.93  0.75        1.13
          2500+      0.91        0.89  0.72        0.90  0.00        1.27
all       all        1.00        1.00  0.98        1.00  0.87        0.99
          <2200      1.03        1.03  1.05        1.04  0.96        1.03
          2200-2499  0.99        0.99  0.91        0.97  0.69        0.93
          2500+      0.87        0.85  0.77        0.78  0.92        1.05

**Model vs data in 50-point bins (n ≥ 100), at each game's own rating**

main                                    interaction                                   
                     bins inside mean_err_pp mean_err_wtd_pp        bins inside mean_err_pp mean_err_wtd_pp
classical <2200      38.0   34.0        0.15            0.20        38.0   34.0       -0.04            0.22
          2200-2499  27.0   17.0       -0.53            0.08        27.0   22.0       -0.08            0.06
          2500+      14.0    5.0       -1.81           -2.03        14.0    5.0       -1.81           -2.16
rapid     <2200      38.0   33.0        0.08            0.02        38.0   35.0       -0.03            0.04
          2200-2499  26.0   25.0        0.02            0.39        26.0   25.0        0.36            0.27
          2500+      12.0    9.0       -0.48           -0.88        12.0   11.0       -0.81           -0.87
blitz     <2200      34.0   31.0        0.24            0.09        34.0   33.0        0.08            0.11
          2200-2499  26.0   26.0       -0.25            0.27        26.0   26.0        0.11            0.29
          2500+      14.0   13.0       -0.49           -0.63        14.0   12.0       -0.64           -0.74

exp_total  obs_total  obs_flags
model       alpha                                 
main        0.20     70671.0      71079      71079
            0.10     11273.0      11114      11114
            0.05      1344.0       1183       1183
            0.01         0.0          0          0
interaction 0.20     70407.0      70202      70202
            0.10     10786.0      10776      10776
            0.05      1410.0       1414       1414
            0.01        10.0          4          4

Among games with P(win) < 0.10, observed/expected wins by band (<2200 / 2200–2499 / 2500+): main model 1.05 / 0.91 / 0.77, interaction model 1.04 / 0.97 / 0.78. Bins inside the 95 % interval: main 193/229, interaction 203/229; classical 2500+ n-weighted mean error -2.0 (main) vs -2.2 points (interaction).

## 10. The ordering check: classical < rapid < blitz

For every underdog colour, strength band and α in {0.20, 0.10, 0.05} (18 cells per run): is the critical gap of classical smaller than that of rapid, and that of rapid smaller than that of blitz? And how many of the three pairwise differences have an interval that excludes 0?

In [20]:
rows = []
for name, suf in RUNS.items():
    c = load_crit(suf); dd = pd.read_csv(os.path.join(OUT, f"critical_gap_diff_empirical{suf}.csv"))
    for (col, band, al), d in c[c.alpha.isin([0.20, 0.10, 0.05])].groupby(["underdog_colour", "band", "alpha"]):
        v = d.set_index("tc_class").critical_gap
        x = dd[(dd.underdog_colour == col) & (dd.band == band) & np.isclose(dd.alpha, al)]
        rows.append(dict(run=name, underdog_colour=col, band=band, alpha=al, classical=v["classical"], rapid=v["rapid"], blitz=v["blitz"],
                         ordering_holds=bool(v["classical"] < v["rapid"] < v["blitz"]), pairs_excluding_0=int(((x.hi95 < 0) | (x.lo95 > 0)).sum()),
                         pairs_not_excluding_0="; ".join(x[~((x.hi95 < 0) | (x.lo95 > 0))].pair)))
oc = pd.DataFrame(rows)
oc.to_csv(os.path.join(OUT, "ordering_check.csv"), index=False)
s = oc.groupby(["run", "alpha"]).agg(cells=("ordering_holds", "size"), ordering_holds=("ordering_holds", "sum"), pairs=("pairs_excluding_0", "size"),
                                      pairs_excluding_0=("pairs_excluding_0", "sum"))
s["pairs"] *= 3
display(s.reindex(list(RUNS), level=0).sort_index(level=1, ascending=False, sort_remaining=False))
ex = oc[oc.pairs_not_excluding_0 != ""]
display(ex[["run", "underdog_colour", "band", "alpha", "classical", "rapid", "blitz", "pairs_not_excluding_0"]].reset_index(drop=True))
say(f"The ordering classical < rapid < blitz holds in **{int(oc.ordering_holds.sum())} of {len(oc)}** cells over the four runs "
    f"({int(oc[oc.run == 'main'].ordering_holds.sum())} of 18 in the main run). The table lists the cells where a difference interval includes 0.")

,,cells,ordering_holds,pairs,pairs_excluding_0
run,alpha,,,,
main,0.20,6,6,18,18
verified,0.20,6,6,18,18
sensitivity,0.20,6,6,18,18
interaction,0.20,6,6,18,18
main,0.10,6,6,18,18
verified,0.10,6,6,18,18
sensitivity,0.10,6,6,18,18
interaction,0.10,6,6,18,17
main,0.05,6,6,18,15


,run,underdog_colour,band,alpha,classical,rapid,blitz,pairs_not_excluding_0
0,main,white,2200-2499,0.05,445.0,625.0,675.0,rapid - blitz
1,main,white,2500+,0.05,400.0,595.0,650.0,rapid - blitz
2,main,white,<2200,0.05,495.0,660.0,700.0,rapid - blitz
3,verified,white,<2200,0.05,445.0,530.0,600.0,rapid - blitz
4,sensitivity,white,2200-2499,0.05,445.0,605.0,650.0,rapid - blitz
5,sensitivity,white,2500+,0.05,400.0,575.0,630.0,rapid - blitz
6,sensitivity,white,<2200,0.05,495.0,640.0,670.0,rapid - blitz
7,interaction,white,2200-2499,0.05,400.0,540.0,580.0,rapid - blitz
8,interaction,white,2500+,0.05,355.0,500.0,530.0,rapid - blitz
9,interaction,white,2500+,0.10,235.0,370.0,410.0,rapid - blitz


The ordering classical < rapid < blitz holds in **72 of 72** cells over the four runs (18 of 18 in the main run). The table lists the cells where a difference interval includes 0.

## 11. The upsets under the final definition

### 11.1 How many?

In [21]:
fl = pd.read_parquet(os.path.join(OUT, "upsets_empirical.parquet"))
g = games.merge(fl, on="game_url", how="left")                       # exact ties get NaN (no underdog)
us = g.underdog_score.to_numpy()
rows = []
for al in ALPHAS:
    f = g[f"upset_emp_{al:.2f}"].fillna(False).astype(bool)
    rows.append({"alpha": al, **{c: int(f[g.tc_class == c].sum()) for c in CLASSES}, **{f"{c} %": 100 * f[(g.tc_class == c) & g.has_underdog].mean() for c in CLASSES},
                 "total": int(f.sum()), "underdog wins": int((f & (us == 1)).sum()), "underdog draws": int((f & (us == 0.5)).sum())})
display(pd.DataFrame(rows).set_index("alpha").round(2))
tail = g[g.has_underdog & (g.gap >= 700)].assign(win=lambda d: d.underdog_score == 1, nl=lambda d: d.underdog_score > 0)
display(tail.groupby(["tc_class", "underdog_color"]).agg(games=("win", "size"), win=("win", "mean"), not_lose=("nl", "mean")).round(4))
n01 = int(g["upset_emp_0.01"].fillna(False).astype(bool).sum())
say((f"No game is an upset at α = 0.01: even at the largest gaps the model gives the underdog more than a 1 % chance of a win or draw. " if n01 == 0 else
     f"Only {n01} games are upsets at α = 0.01. ") + "The second table shows the raw rates at gaps of 700 and more (small numbers).")

,classical,rapid,blitz,classical %,rapid %,blitz %,total,underdog wins,underdog draws
alpha,,,,,,,,,
0.20,55738,10322,5019,9.10,7.24,4.68,71079,64928,6151
0.10,9007,1483,624,1.47,1.04,0.58,11114,10510,604
0.05,977,150,56,0.16,0.11,0.05,1183,1161,22
0.01,0,0,0,0.00,0.00,0.00,0,0,0


games     win  not_lose
tc_class  underdog_color                         
blitz     black             687  0.0495    0.0670
          white             679  0.0457    0.0722
classical black            1414  0.0191    0.0552
          white            1307  0.0230    0.0735
rapid     black            1193  0.0419    0.1031
          white            1110  0.0423    0.0892

No game is an upset at α = 0.01: even at the largest gaps the model gives the underdog more than a 1 % chance of a win or draw. The second table shows the raw rates at gaps of 700 and more (small numbers).

### 11.2 Are the upsets real games? (audit at α = 0.10)

Upsets could pile up in suspicious games: walkovers recorded as games, wrong ratings, mis-classified time controls. The table compares the upsets with all games; the 200 rarest upsets are saved with their Lichess link (`upset_audit_top.csv`) so that they can be checked by hand.

In [22]:
up = g[g["upset_emp_0.10"].fillna(False).astype(bool)].copy()
au = pd.DataFrame({"upsets (α = 0.10)": [up.short_game.mean(), up.elo_mismatch_any.fillna(False).mean(), up.tc_borderline.fillna(False).mean(), up.verified.mean()],
                   "all games": [g.short_game.mean(), g.elo_mismatch_any.fillna(False).mean(), g.tc_borderline.fillna(False).mean(), g.verified.mean()]},
                  index=["short game (< 20 half-moves)", "PGN rating differs from FIDE by > 100", "time control near a class boundary", "both ratings verified"])
display((100 * au).round(1))
top = up.sort_values("q_emp").head(200)
cols = [c for c in ("game_url", "broadcast_name", "round", "white", "black", "white_elo_final", "black_elo_final", "gap", "score_white",
                    "tc_class", "n_plies", "termination", "q_emp", "short_game", "elo_mismatch_any") if c in top.columns]
top[cols].to_csv(os.path.join(OUT, "upset_audit_top.csv"), index=False)
display(top[["broadcast_name", "white", "black", "white_elo_final", "black_elo_final", "score_white", "tc_class", "n_plies", "q_emp"]].head(5))
cmpw = lambda i: "more" if au.iloc[i, 0] > au.iloc[i, 1] else "less"
say(f"**{len(up):,}** upsets at α = 0.10. Compared with all games they are {cmpw(0)} often short games ({100 * au.iloc[0, 0]:.1f} % vs {100 * au.iloc[0, 1]:.1f} %), "
    f"{cmpw(1)} often games with a PGN rating that disagrees with FIDE ({100 * au.iloc[1, 0]:.1f} % vs {100 * au.iloc[1, 1]:.1f} %), "
    f"and {cmpw(3)} often games with both ratings verified ({100 * au.iloc[3, 0]:.1f} % vs {100 * au.iloc[3, 1]:.1f} %): "
    "an unverified rating makes an underdog look weaker than it is, which is why the verified-ratings run of section 8 matters.")

,upsets (α = 0.10),all games
short game (< 20 half-moves),0.2,0.7
PGN rating differs from FIDE by > 100,1.8,2.0
time control near a class boundary,2.7,3.9
both ratings verified,63.6,76.1


,broadcast_name,white,black,white_elo_final,black_elo_final,score_white,tc_class,n_plies,q_emp
148785,Nori Open,Yash Krishen Maharaj,Johan-Sebastian Christiansen,1537,2605,1.0,classical,83,0.013571
749449,Israel National Team Championship | Bottom Pla...,"Kovalev, Vladislav","Gershon, Alik",1455,2426,1.0,classical,71,0.015036
77106,44th Chess Olympiad 2022 | Open | Boards 41-168,Edwards Emar,Westerberg Jonathan,1695,2504,1.0,classical,59,0.016748
69508,Pan American Youth Chess Festival,"Navarro Ariza, Justin Jeshua","Castro Silva, Moises Antonio",1290,2237,1.0,classical,77,0.017276
728014,3rd Shri Dhanpat Rai Sachdeva Memorial Interna...,"Erenberg, Ariel",Ishan Ghosh (Siliguri),2487,1769,0.0,classical,90,0.017308


**11,114** upsets at α = 0.10. Compared with all games they are less often short games (0.2 % vs 0.7 %), less often games with a PGN rating that disagrees with FIDE (1.8 % vs 2.0 %), and less often games with both ratings verified (63.6 % vs 76.1 %): an unverified rating makes an underdog look weaker than it is, which is why the verified-ratings run of section 8 matters.

### 11.3 Repeat giant-killers

One player behind many upsets is either a rating error (a junior improving faster than the rating list) or a genuine giant-killer. For every player with at least 30 games as the underdog: number of upsets, upset rate and a robust z-score (median and MAD of the rates). Players are identified by FIDE id, by name when the id is missing or a placeholder (`0`, `-`, etc.). Sections 11.1-11.3 build on each other and are re-run together.

In [23]:
up["underdog_name"] = np.where(up.underdog_color == "white", up.white, up.black)
# placeholder ids ('0', '-', '-1', ...) are shared by many different players: they are treated as missing, so the name is used instead
real_id = lambda s: s.where(s.astype("string").str.fullmatch(r"[1-9]\d*").fillna(False).astype(bool))
up["underdog_fide"] = np.where(up.underdog_color == "white", real_id(up.white_fide_id), real_id(up.black_fide_id))
key = up["underdog_fide"].fillna(up["underdog_name"])
all_u = g[g.has_underdog].copy()
all_u["key"] = np.where(all_u.underdog_color == "white", real_id(all_u.white_fide_id), real_id(all_u.black_fide_id))
all_u["key"] = pd.Series(all_u["key"]).fillna(pd.Series(np.where(all_u.underdog_color == "white", all_u.white, all_u.black), index=all_u.index))
gk = pd.DataFrame({"games_as_underdog": all_u.groupby("key").size(), "upsets": key.value_counts()}).dropna()
gk = gk[gk.games_as_underdog >= 30]
gk["upset_rate"] = gk.upsets / gk.games_as_underdog
med = gk.upset_rate.median(); mad = (gk.upset_rate - med).abs().median() * 1.4826
gk["robust_z"] = (gk.upset_rate - med) / max(mad, 1e-9)
gk.index.name = None
gk = gk.sort_values(["upsets", "upset_rate"], ascending=False, kind="mergesort")      # stable: ties keep a fixed order
gk.head(300).to_csv(os.path.join(OUT, "giant_killers.csv"))
display(gk.head(10).round(3))
say(f"**{len(gk):,}** players have at least 30 games as the underdog; their median upset rate is {100 * med:.2f} %. "
    f"{int((gk.robust_z > 3.5).sum())} have a robust z above 3.5; the 20 players with the most upsets account for {100 * gk.upsets.head(20).sum() / len(up):.1f} % of all upsets.")
del g, up, all_u

,games_as_underdog,upsets,upset_rate,robust_z
13816063,54,13.0,0.241,17.272
13752260,144,13.0,0.090,5.190
45075204,80,12.0,0.150,9.986
13810251,81,10.0,0.123,7.855
11205385,45,9.0,0.200,14.001
16701160,47,9.0,0.191,13.318
9932470,83,9.0,0.108,6.648
4908120,99,9.0,0.091,5.241
8641528,113,9.0,0.080,4.337
45039895,118,9.0,0.076,4.066


**2,206** players have at least 30 games as the underdog; their median upset rate is 2.56 %. 125 have a robust z above 3.5; the 20 players with the most upsets account for 1.6 % of all upsets.

### 11.4 Win / draw / loss counts by gap bin (for the figures and the dashboard)

In [24]:
cbg = games[games.has_underdog].assign(u_win=lambda d: (d.underdog_score == 1).astype(int), u_draw=lambda d: (d.underdog_score == 0.5).astype(int),
                                       u_loss=lambda d: (d.underdog_score == 0).astype(int))
cb = (cbg.groupby(["tc_class", "underdog_color", "band", "verified", "gap_bin"])
         .agg(games=("u_win", "size"), wins=("u_win", "sum"), draws=("u_draw", "sum"), losses=("u_loss", "sum")).reset_index())
cb.to_csv(os.path.join(OUT, "curve_bins.csv"), index=False)
assert cb.games.sum() == len(cbg)
say(f"`curve_bins.csv`: {len(cb)} rows (time control × colour × band × verified flag × 50-point gap bin, 600 = 600+), {int(cb.games.sum()):,} games; "
    "all games are kept, the `verified` column tells the two groups apart.")
del cbg

`curve_bins.csv`: 448 rows (time control × colour × band × verified flag × 50-point gap bin, 600 = 600+), 862,599 games; all games are kept, the `verified` column tells the two groups apart.

## 12. Output files

In [25]:
rows = []
for f in sorted(os.listdir(OUT)):
    p = os.path.join(OUT, f)
    if f == "analysis_games.parquet" or not f.endswith((".csv", ".parquet")): continue
    n, k = (pq.ParquetFile(p).metadata.num_rows, pq.ParquetFile(p).metadata.num_columns) if f.endswith(".parquet") else (len(pd.read_csv(p)), pd.read_csv(p, nrows=1).shape[1])
    rows.append(dict(file=f, rows=n, columns=k, KB=round(os.path.getsize(p) / 1e3)))
    if f.endswith(".csv"): shutil.copy(p, os.path.join(SMALL_OUT, f))
display(pd.DataFrame(rows).set_index("file"))
say(f"The CSV files are also copied to `{os.path.relpath(SMALL_OUT)}` (with the text reports `empirical_report*.txt`). Notebook **04_report** draws the figures from these files.")

,rows,columns,KB
file,,,
calibration_empirical.csv,48,8,5
calibration_empirical_interaction.csv,48,8,5
calibration_empirical_sens.csv,48,8,5
calibration_empirical_verified.csv,48,8,5
critical_gap_decisive.csv,18,6,1
critical_gap_decisive_sens.csv,18,6,1
critical_gap_decisive_verified.csv,18,6,1
critical_gap_diff_empirical.csv,54,7,3
critical_gap_diff_empirical_interaction.csv,70,7,4


The CSV files are also copied to `..\reports\model_outputs` (with the text reports `empirical_report*.txt`). Notebook **04_report** draws the figures from these files.